# Alexander von Humboldt – Amerikanische Reisetagebücher III: Digitale Edition

Processes Humboldt's digitized journal page scans through the full pipeline:

**Region Detection → Transcription → Consistency Check → NER → Geocoding → HTML Edition**

---

##  Upload images


In [ ]:
import os
from pathlib import Path
from google.colab import files

# Upload the page image scans
print("Please upload your page image scans:")
uploaded_images_dict = files.upload()
image_files = list(uploaded_images_dict.keys())

# --- Summary ---
print("\n" + "="*30)
print(f"✓ Image files uploaded: {len(image_files)} files")
print("="*30)

Please upload your page image scans:


Saving H1242132__34r.jpg to H1242132__34r.jpg
Saving H1242132__33v.jpg to H1242132__33v.jpg
Saving H1242132__33r.jpg to H1242132__33r.jpg
Saving H1242132__32v.jpg to H1242132__32v.jpg

✓ Image files uploaded: 4 files


## Clone the project & install dependencies

In [ ]:
!git clone https://github.com/Maelkolb/Humboldt_The_Inaccurate_Edition /content/humboldt-edition

PROJECT_DIR = Path("/content/humboldt-edition")
print(f"\n✓ Project cloned to {PROJECT_DIR}")
!ls {PROJECT_DIR}/src/

Cloning into '/content/humboldt-edition'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 78 (delta 41), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 68.35 KiB | 4.02 MiB/s, done.
Resolving deltas: 100% (41/41), done.

✓ Project cloned to /content/humboldt-edition
config.py	      html_generator.py  ner.py		      transcription.py
consistency_check.py  __init__.py	 pipeline.py
downloader.py	      json_utils.py	 region_detection.py
geocoding.py	      models.py		 tei_parser.py


In [ ]:
!pip install -q -r {PROJECT_DIR}/requirements.txt
!pip install -q lxml   # needed for TEI evaluation

## Move images into the project's images/ folder

In [ ]:
IMAGE_FOLDER = PROJECT_DIR / "images"
IMAGE_FOLDER.mkdir(exist_ok=True)

import shutil
valid_ext = {".jpg", ".jpeg", ".png", ".tiff", ".tif", ".bmp"}
for name in image_files:
    src = Path(f"/content/{name}")
    if src.suffix.lower() in valid_ext:
        shutil.copy2(src, IMAGE_FOLDER / name)

imgs = sorted(IMAGE_FOLDER.iterdir())
print(f"✓ {len(imgs)} images in {IMAGE_FOLDER}:")
for f in imgs:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

✓ 4 images in /content/humboldt-edition/images:
  H1242132__32v.jpg  (1.3 MB)
  H1242132__33r.jpg  (1.1 MB)
  H1242132__33v.jpg  (1.2 MB)
  H1242132__34r.jpg  (1.2 MB)


## Set Gemini API Key - colab secret

In [ ]:
# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✓ API key loaded from Colab Secrets")
except Exception:
    pass

# Option B: Paste directly (uncomment)
# os.environ["GEMINI_API_KEY"] = "your-api-key-here"

assert os.environ.get("GEMINI_API_KEY"), "Set GEMINI_API_KEY via Secrets or Option B above"

✓ API key loaded from Colab Secrets


## Import the project codebase

In [ ]:
import sys
import logging

sys.path.insert(0, str(PROJECT_DIR))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)

from src import config, process_book, generate_html_edition, load_results_from_json
from google import genai

print(f"✓ Project imported")
print(f"  Model:    {config.MODEL_ID}")
print(f"  Entities: {len(config.ENTITY_TYPES)} types: {list(config.ENTITY_TYPES)}")
print(f"  Regions:  {len(config.REGION_TYPES)} types")

OUTPUT_FOLDER = PROJECT_DIR / "output"
OUTPUT_FOLDER.mkdir(exist_ok=True)

✓ Project imported
  Model:    gemini-3-flash-preview
  Entities: 8 types: ['Person', 'Location', 'Indigenous_Group', 'Instrument', 'Species', 'Publication', 'Celestial_Object', 'Measurement']
  Regions:  15 types


## Model/run Settings

In [ ]:
# ── SETTINGS ──────────────────────────────────────────────────────────────

MAX_PAGES = None    # None = all pages; set e.g. 2 to test on first 2 folios
THINKING  = "high" # "low" | "medium" | "high" — higher = slower but better

# Runs a post-transcription LLM check that detects and fixes:
#   - Duplicate lines (marginal note text accidentally copied into main text)
#   - Language label mismatches
#   - Empty regions that should have content
# Adds ~5 s per folio. Only disable if you need maximum speed.
RUN_CONSISTENCY_CHECK = True

## Run the pipeline

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print(f"Gemini client ready\n")

results = process_book(
    client=client,
    image_folder=str(IMAGE_FOLDER),
    output_folder=str(OUTPUT_FOLDER),
    entity_types=config.ENTITY_TYPES,
    model_id=config.MODEL_ID,
    thinking_level=THINKING,
    run_consistency_check=RUN_CONSISTENCY_CHECK,
    start_page=None,
    end_page=MAX_PAGES,
)

print(f"\n{'='*60}")
print(f"✓ Pipeline complete!")
print(f"  Folios processed: {len(results)}")
print(f"  Total entities:   {sum(len(r.entities) for r in results)}")
print(f"  Total locations:  {sum(len(r.locations) for r in results)}")
print(f"  Marginal notes:   {sum(1 for r in results for reg in r.regions if reg.region_type == 'marginal_note')}")
print(f"  Pasted slips:     {sum(1 for r in results for reg in r.regions if reg.region_type == 'pasted_slip')}")
print(f"  Usage marks:      {sum(1 for r in results for reg in r.regions if reg.region_type == 'usage_mark')}")

Gemini client ready



Folios: 100%|██████████| 4/4 [05:56<00:00, 89.08s/fol]


✓ Pipeline complete!
  Folios processed: 4
  Total entities:   113
  Total locations:  48
  Marginal notes:   12
  Pasted slips:     2
  Usage marks:      1


## Step 7 – Generate the HTML edition

In [ ]:
html_path = generate_html_edition(
    results=results,
    output_path=OUTPUT_FOLDER / "humboldt_edition.html",
    title="Alexander v. Humboldt – The Inaccurate Edition",
    subtitle="Amerikanische Reisetagebücher III",
    entity_colors=config.ENTITY_COLORS,
    entity_labels=config.ENTITY_LABELS,
    region_colors=config.REGION_COLORS,
    region_labels=config.REGION_LABELS,
    image_folder=str(IMAGE_FOLDER),   # embeds facsimile images as base64
)

print(f"\n✓ Edition ready: {html_path}")
print(f"  File size: {html_path.stat().st_size / 1e6:.1f} MB")


✓ Edition ready: /content/humboldt-edition/output/humboldt_edition.html
  File size: 1.6 MB


## Download

In [ ]:
from google.colab import files

# HTML edition
files.download(str(OUTPUT_FOLDER / "humboldt_edition.html"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# JSON data (save this — lets you regenerate the HTML without re-running the pipeline)
files.download(str(OUTPUT_FOLDER / "digital_edition_complete.json"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Evaluation – Compare pipeline output against TEI ground truth



This compares:
- **Text accuracy** — CER (Character Error Rate) and WER (Word Error Rate) per folio
- **Region recall** — which region types the pipeline correctly detected
- **Marginal note recall** — how many of the TEI `<note place="...">` elements were found
- **Entity precision/recall** — against TEI `<persName>`, `<placeName>`, `<orgName>`

In [ ]:
print("Upload the TEI XML file for evaluation:")
eval_uploaded = files.upload()
tei_files = [n for n in eval_uploaded if n.endswith(".xml")]
print(f"✓ Uploaded: {tei_files}")

In [ ]:
from src.tei_parser import parse_tei_file

tei_path = f"/content/{tei_files[0]}"
tei_results = parse_tei_file(tei_path)
print(f"✓ TEI parsed: {len(tei_results)} pages as ground truth")

# ── Evaluation metrics ──────────────────────────────────────────────────
import unicodedata, re

def _normalise(text):
    """Lowercase, strip editorial markup and punctuation for fair comparison."""
    text = re.sub(r'~~.+?~~', '', text)          # remove ~~strikethrough~~
    text = re.sub(r'\[\?\]', '', text)            # remove [?] markers
    text = re.sub(r'\[\.\.\.]', '', text)         # remove [...] gaps
    text = re.sub(r'\[([^\]]+)\]', r'\1', text)  # keep [supplied] text
    text = unicodedata.normalize('NFC', text.lower())
    text = re.sub(r'[^\w\s]', ' ', text)
    return ' '.join(text.split())

def cer(ref, hyp):
    """Character Error Rate (edit distance / len(ref))."""
    r, h = list(ref), list(hyp)
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = d[i-1][j-1] if r[i-1]==h[j-1] else 1 + min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[len(r)][len(h)] / max(len(r), 1)

# Match pages by folio label
tei_by_folio = {r.folio_label: r for r in tei_results}
pip_by_folio = {r.folio_label: r for r in results}

common = sorted(set(tei_by_folio) & set(pip_by_folio))
print(f"\nMatched {len(common)} folios for evaluation:\n")

cer_scores, margin_recall_scores = [], []

print(f"{'Folio':8s}  {'CER':>7s}  {'Margin recall':>14s}  {'Slips found':>11s}")
print("-" * 50)

for folio in common:
    tei_r = tei_by_folio[folio]
    pip_r = pip_by_folio[folio]

    # CER on main text
    ref_text = _normalise(tei_r.full_text)
    hyp_text = _normalise(pip_r.full_text)
    c = cer(ref_text, hyp_text)
    cer_scores.append(c)

    # Marginal note recall: how many TEI margin notes were detected?
    tei_margins = [r for r in tei_r.regions if r.region_type == "marginal_note"]
    pip_margins = [r for r in pip_r.regions if r.region_type == "marginal_note"]
    # Simple heuristic: check if any TEI margin text appears in any pipeline margin
    found = 0
    for tm in tei_margins:
        words = set(_normalise(tm.content).split()[:5])  # first 5 words
        for pm in pip_margins:
            if words & set(_normalise(pm.content).split()):
                found += 1
                break
    recall = found / max(len(tei_margins), 1)
    margin_recall_scores.append(recall)

    # Pasted slips
    tei_slips = sum(1 for r in tei_r.regions if r.region_type == "pasted_slip")
    pip_slips = sum(1 for r in pip_r.regions if r.region_type == "pasted_slip")

    print(f"{folio:8s}  {c:7.1%}  {recall:14.0%}  {pip_slips}/{tei_slips} slips")

print("-" * 50)
print(f"{'AVERAGE':8s}  {sum(cer_scores)/len(cer_scores):7.1%}  "
      f"{sum(margin_recall_scores)/len(margin_recall_scores):14.0%}")
print()
print("CER = Character Error Rate (lower is better; 0% = perfect match)")
print("Margin recall = fraction of TEI margin notes found by the pipeline")

In [ ]:
# ── Entity evaluation ────────────────────────────────────────────────────
# Compares pipeline NER output against TEI-tagged entities (persName, placeName, orgName)

TYPE_MAP = {"Person": "Person", "Location": "Location", "Institution": "Institution"}

for folio in common:
    tei_r = tei_by_folio[folio]
    pip_r = pip_by_folio[folio]

    tei_ents = {(_normalise(e.text), e.entity_type) for e in tei_r.entities
                if e.entity_type in TYPE_MAP}
    pip_ents = {(_normalise(e.text), e.entity_type) for e in pip_r.entities
                if e.entity_type in TYPE_MAP}

    tp = len(tei_ents & pip_ents)
    fp = len(pip_ents - tei_ents)
    fn = len(tei_ents - pip_ents)
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)

    if tei_ents or pip_ents:
        print(f"Fol. {folio:6s}  P={prec:.0%}  R={rec:.0%}  F1={f1:.0%}  "
              f"(TP={tp}, FP={fp}, FN={fn})")

---
## Extras

### Re-generate HTML from saved JSON (no API calls needed)

In [ ]:
# results = load_results_from_json(OUTPUT_FOLDER / "digital_edition_complete.json")
#
# html_path = generate_html_edition(
#     results=results,
#     output_path=OUTPUT_FOLDER / "humboldt_edition_v2.html",
#     title="Alexander von Humboldt – Amerikanische Reisetagebücher",
#     subtitle="Digitale Edition v2",
#     entity_colors=config.ENTITY_COLORS,
#     entity_labels=config.ENTITY_LABELS,
#     region_colors=config.REGION_COLORS,
#     region_labels=config.REGION_LABELS,
#     image_folder=str(IMAGE_FOLDER),
# )

### Inspect a single folio result

In [ ]:
if results:
    r = results[0]
    print(f"Folio:     {r.folio_label}")
    print(f"Entries:   {r.entry_numbers}")
    print(f"Languages: {r.page_languages}")
    print(f"Regions:   {len(r.regions)}")
    print(f"Entities:  {len(r.entities)}")
    print(f"Locations: {len(r.locations)}")
    print(f"\n--- Regions ---")
    for reg in r.regions:
        preview = reg.content[:80].replace("\n", " ") + ("..." if len(reg.content) > 80 else "")
        mp   = f" [{reg.marginal_position}]" if reg.marginal_position else ""
        slip = " [SLIP]" if reg.is_pasted_slip else ""
        used = " [USED]" if reg.is_usage_marked else ""
        print(f"  [{reg.region_type}{mp}{slip}{used}] {preview}")
    print(f"\n--- Entities (first 20) ---")
    for e in r.entities[:20]:
        norm = f" → {e.normalized_form}" if e.normalized_form else ""
        print(f"  [{e.entity_type}] {e.text}{norm}")

### Run consistency check on existing results

If you loaded results from a previous JSON run and want to apply the
deduplication/consistency check without re-running the full pipeline:

In [ ]:
# from src.consistency_check import check_and_fix_regions
#
# for r in results:
#     r.regions, issues = check_and_fix_regions(
#         client, r.regions, config.MODEL_ID, thinking_level="low"
#     )
#     if issues:
#         print(f"Fol. {r.folio_label}: {len(issues)} issue(s) corrected")